# 深度 Q 学习——月球着陆器

在本次作业中，你将训练一个智能体，让月球着陆器安全降落在月球表面的着陆平台上。


# 目录
- [ 1 - 导入程序包 <img align="Right" src="./图片/lunar_lander.gif" width = 60% >](#1)
- [ 2 - 超参数](#2)
- [ 3 - 月球着陆器环境](#3)
  - [ 3.1 动作空间](#3.1)
  - [ 3.2 观测空间](#3.2)
  - [ 3.3 奖励](#3.3)
  - [ 3.4 回合终止](#3.4)
- [ 4 - 加载环境](#4)
- [ 5 - 与 Gym 环境交互](#5)
    - [ 5.1 探索环境的动态特性](#5.1)
- [ 6 - 深度 Q 学习](#6)
  - [ 6.1 目标网络](#6.1)
    - [ 练习 1](#ex01)
  - [ 6.2 经验回放](#6.2)
- [ 7 - 带经验回放的深度 Q 学习算法](#7)
  - [ 练习 2](#ex02)
- [ 8 - 更新网络权重](#8)
- [ 9 - 训练智能体](#9)
- [ 10 - 观察训练后的智能体](#10)
- [ 11 - 恭喜！](#11)
- [ 12 - 参考文献](#12)


<a name="1"></a>
## 1 - 导入程序包

我们将使用以下程序包：
- `numpy` 是 Python 中用于科学计算的程序包。
- `deque` 将作为我们记忆缓冲区的数据结构。
- `namedtuple` 将用于存储经验元组。
- `gym` 工具包是一组可用于测试强化学习算法的环境。需要注意，在本 notebook 中我们使用的 `gym` 版本是 `0.24.0`。
- `PIL.Image` 和 `pyvirtualdisplay` 用于渲染月球着陆器环境。
- 我们将使用 `tensorflow.keras` 框架中的若干模块来构建深度学习模型。
- `utils` 是包含本次作业辅助函数的模块。你不需要修改该文件中的代码。

运行下面的单元格，导入所有必要的程序包。

In [ ]:
import time
from collections import deque, namedtuple

import gym
import numpy as np
import PIL.Image
import tensorflow as tf
import utils

from pyvirtualdisplay import Display
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import MSE
from tensorflow.keras.optimizers import Adam

In [ ]:
# 设置虚拟显示器以渲染月球着陆器环境。
Display(visible=0, size=(840, 480)).start();

# 设置 TensorFlow 的随机种子
tf.random.set_seed(utils.SEED)

<a name="2"></a>
## 2 - 超参数

运行下面的单元格以设置超参数。

In [ ]:
MEMORY_SIZE = 100_000     # 记忆缓冲区大小
GAMMA = 0.995             # 折扣因子
ALPHA = 1e-3              # 学习率
NUM_STEPS_FOR_UPDATE = 4  # 每 C 个时间步执行一次学习更新

<a name="3"></a>
## 3 - 月球着陆器环境

在本 notebook 中，我们将使用 [OpenAI 的 Gym 库](https://www.gymlibrary.ml/)。Gym 库提供了种类丰富的强化学习环境。简单来说，一个环境代表一个待解决的问题或任务。在本 notebook 中，我们将尝试用强化学习解决月球着陆器环境。

月球着陆器环境的目标是让着陆器安全降落在月球表面的着陆平台上。着陆平台由两根旗杆标出，它始终位于坐标 `(0,0)`，但也允许着陆器降落在平台之外。着陆器从环境的顶部中央开始，其质心会被施加一个随机的初始力，并且燃料无限。如果得到 `200` 分，就认为该环境已被解决。

<br>
<br>
<figure>
  <img src = "图片/lunar_lander.gif" width = 40%>
      <figcaption style = "text-align: center; font-style: italic">图 1. 月球着陆器环境。</figcaption>
</figure>



<a name="3.1"></a>
### 3.1 动作空间

智能体有四个离散动作可选：

* 什么都不做。
* 点燃右侧引擎。
* 点燃主引擎。
* 点燃左侧引擎。

每个动作对应一个数值：


```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```


<a name="3.2"></a>
### 3.2 观测空间

智能体的观测空间由一个含 8 个变量的状态向量构成：

* 其 $(x,y)$ 坐标。着陆平台始终位于坐标 $(0,0)$。
* 其线速度 $(\dot x,\dot y)$。
* 其角度 $\theta$。
* 其角速度 $\dot \theta$。
* 两个布尔量 $l$ 和 $r$，表示每条腿是否与地面接触。

<a name="3.3"></a>
### 3.3 奖励

月球着陆器环境采用如下奖励机制：

* 降落在着陆平台上并停稳大约得 100-140 分。
* 若着陆器远离着陆平台，会损失奖励。
* 若着陆器坠毁，得 -100 分。
* 若着陆器停稳，得 +100 分。
* 每条与地面接触的腿得 +10 分。
* 点燃主引擎每帧 -0.3 分。
* 点燃侧向引擎每帧 -0.03 分。

<a name="3.4"></a>
### 3.4 回合终止

出现以下情况时，一个回合结束（即环境进入终止状态）：

* 月球着陆器坠毁（即着陆器机身与月球表面接触）。

* 着陆器的 $x$ 坐标大于 1。

关于该环境的完整说明，可查阅 [Open AI Gym 文档](https://www.gymlibrary.ml/environments/box2d/lunar_lander/)。

<a name="4"></a>
## 4 - 加载环境

我们先使用 `.make()` 方法从 `gym` 库加载 `LunarLander-v2` 环境。`LunarLander-v2` 是月球着陆器环境的最新版本，其版本历史可在 [Open AI Gym 文档](https://www.gymlibrary.ml/environments/box2d/lunar_lander/#version-history)中查阅。

In [ ]:
env = gym.make('LunarLander-v2')

加载环境后，我们用 `.reset()` 方法把环境重置到初始状态。着陆器从环境顶部中央开始，我们可以用 `.render()` 方法渲染环境的第一帧。

In [ ]:
env.reset()
PIL.Image.fromarray(env.render(mode='rgb_array'))

为了稍后构建神经网络，我们需要知道状态向量的长度和有效动作的数量。我们可以分别用 `.observation_space.shape` 和 `action_space.n` 方法从环境中获取这些信息。

In [ ]:
state_size = env.observation_space.shape
num_actions = env.action_space.n

print('State Shape:', state_size)
print('Number of actions:', num_actions)

<a name="5"></a>
## 5 - 与 Gym 环境交互

Gym 库实现了标准的“智能体—环境循环”范式：

<br>
<center>
<video src = "./视频/rl_formalism.m4v" width="840" height="480" controls autoplay loop poster="./图片/rl_formalism.png"> </video>
<figcaption style = "text-align:center; font-style:italic">图 2. 智能体—环境循环范式。</figcaption>
</center>
<br>

在标准的“智能体—环境循环”范式中，智能体在离散时间步 $t=0,1,2,...$ 与环境交互。在每个时间步 $t$，智能体依据对环境状态 $S_t$ 的观测，用策略 $\pi$ 选择一个动作 $A_t$。智能体获得数值奖励 $R_t$，并在下一个时间步转移到新状态 $S_{t+1}$。

<a name="5.1"></a>
### 5.1 探索环境的动态特性

在 Open AI 的 Gym 环境中，我们用 `.step()` 方法运行环境动态特性的单个时间步。在我们使用的 `gym` 版本中，`.step()` 方法接受一个动作并返回四个值：

* `observation`（**对象**）：一个与环境相关的对象，表示你对环境的观测。在月球着陆器环境中，它对应一个 numpy 数组，包含 3.2 观测空间一节所述的着陆器位置和速度。


* `reward`（**浮点数**）：因采取给定动作而返回的奖励量。在月球着陆器环境中，它对应一个 `numpy.float64` 类型的浮点数，如 3.3 奖励一节所述。


* `done`（**布尔值**）：当 done 为 `True` 时，表示本回合已终止，该重置环境了。


* `info`（**字典**）：用于调试的诊断信息。本 notebook 中不会用到该变量，此处列出仅为完整起见。

要开始一个回合，我们需要把环境重置到初始状态。我们用 `.reset()` 方法来完成。


In [ ]:
# 重置环境并获取初始状态。
initial_state = env.reset()

环境重置后，智能体就可以用 `.step()` 方法开始在环境中采取动作。注意，智能体每个时间步只能采取一个动作。

在下面的单元格中，你可以选择不同的动作，观察返回值如何随所选动作变化。记住，在这个环境中智能体有四个离散动作可选，我们在代码中用它们对应的数值来指定：


```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```


In [ ]:
# 选择一个动作
action = 0

# 用给定动作运行一步环境动态。
next_state, reward, done, info = env.step(action)

with np.printoptions(formatter={'float': '{:.3f}'.format}):
    print("Initial State:", initial_state)
    print("Action:", action)
    print("Next State:", next_state)
    print("Reward Received:", reward)
    print("Episode Terminated:", done)
    print("Info:", info)

实践中，训练智能体时我们会用循环让智能体在一个回合内连续采取许多动作。

<a name="6"></a>
## 6 - 深度 Q 学习

当状态空间和动作空间都是离散的时，我们可以用贝尔曼方程迭代地估计动作价值函数：

$$
Q_{i+1}(s,a) = R + \gamma \max_{a'}Q_i(s',a')
$$

随着 $i\to\infty$，这种迭代方法会收敛到最优动作价值函数 $Q^*(s,a)$。这意味着智能体只需逐步探索状态-动作空间，并不断更新 $Q(s,a)$ 的估计，直到它收敛到最优动作价值函数 $Q^*(s,a)$。然而，当状态空间连续时，要探索整个状态-动作空间实际上是不可能的。因此，也就无法逐步估计 $Q(s,a)$ 直到它收敛到 $Q^*(s,a)$。

在深度 $Q$ 学习中，我们通过用神经网络来估计动作价值函数 $Q(s,a)\approx Q^*(s,a)$ 解决这一问题。我们把这个神经网络称为 $Q$ 网络，它可以这样训练：每次迭代调整其权重，以最小化贝尔曼方程中的均方误差。

遗憾的是，在强化学习中使用神经网络估计动作价值函数被证明非常不稳定。幸运的是，有几种技术可以用来避免不稳定。这些技术包括使用***目标网络（Target Network）***和***经验回放（Experience Replay）***。我们将在下面几节中探讨这两种技术。

<a name="6.1"></a>
### 6.1 目标网络

我们可以通过每次迭代调整 $Q$ 网络的权重来训练它，以最小化贝尔曼方程中的均方误差，其中目标值由下式给出：

$$
y = R + \gamma \max_{a'}Q(s',a';w)
$$

其中 $w$ 是 $Q$ 网络的权重。这意味着我们每次迭代都在调整权重 $w$，以最小化下面的误差：

$$
\overbrace{\underbrace{R + \gamma \max_{a'}Q(s',a'; w)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

注意，这会带来一个问题：$y$ 目标在每次迭代中都在变化。目标不断移动可能导致振荡和不稳定。为避免这一点，我们可以创建一个单独的神经网络来生成 $y$ 目标。我们把这个单独的神经网络称为**目标 $\hat Q$ 网络**，它与原 $Q$ 网络结构相同。使用目标 $\hat Q$ 网络后，上面的误差变为：

$$
\overbrace{\underbrace{R + \gamma \max_{a'}\hat{Q}(s',a'; w^-)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

其中 $w^-$ 和 $w$ 分别是目标 $\hat Q$ 网络和 $Q$ 网络的权重。

实践中，我们将使用如下算法：每 $C$ 个时间步，我们用 $\hat Q$ 网络生成 $y$ 目标，并用 $Q$ 网络的权重更新目标 $\hat Q$ 网络的权重。我们将用**软更新（soft update）**更新目标 $\hat Q$ 网络的权重 $w^-$。这意味着我们按下面的规则更新权重 $w^-$：

$$
w^-\leftarrow \tau w + (1 - \tau) w^-
$$

其中 $\tau\ll 1$。通过软更新，我们确保目标值 $y$ 变化缓慢，这大大提高了学习算法的稳定性。

<a name="ex01"></a>
### 练习 1

在本练习中，你将创建 $Q$ 网络和目标 $\hat Q$ 网络，并设置优化器。记住，深度 $Q$ 网络（DQN）是一个近似动作价值函数 $Q(s,a)\approx Q^*(s,a)$ 的神经网络。它通过学习如何把状态映射为 $Q$ 值来做到这一点。

为解决月球着陆器环境，我们将采用具有以下结构的 DQN：

* 一个 `Input` 层，以 `state_size` 为输入。

* 一个 `Dense` 层，含 `64` 个单元，激活函数为 `relu`。

* 一个 `Dense` 层，含 `64` 个单元，激活函数为 `relu`。

* 一个 `Dense` 层，含 `num_actions` 个单元，激活函数为 `linear`。这将是网络的输出层。


在下面的单元格中，你应使用上述模型结构创建 $Q$ 网络和目标 $\hat Q$ 网络。记住，$Q$ 网络和目标 $\hat Q$ 网络的结构相同。

最后，你应把 `Adam` 设为优化器，学习率等于 `ALPHA`。回忆一下，`ALPHA` 在[超参数](#2)一节中已定义。需要注意，本练习中你应使用已经导入的程序包：

```python
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
```


In [ ]:
# UNQ_C1
# 计分单元

# 创建 Q 网络
q_network = Sequential([
    ### 在此处编写代码 ###

    ### 代码编写结束 ###
    ])

# 创建目标 Q^- 网络
target_q_network = Sequential([
    ### 在此处编写代码 ###

    ### 代码编写结束 ###
    ])

### 在此处编写代码 ###
optimizer = None
### 代码编写结束 ###

In [ ]:
# 单元测试
from public_tests import *

test_network(q_network)
test_network(target_q_network)
test_optimizer(optimizer, ALPHA) 

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    

```python
# Create the Q-Network
q_network = Sequential([
    Input(shape=state_size),                      
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'),
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    Input(shape=state_size),                       
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'), 
    ])

optimizer = Adam(learning_rate=ALPHA)                                  
```
 

<a name="6.2"></a>
### 6.2 经验回放

智能体与环境交互时，它所经历的状态、动作和奖励本质上是按时间顺序排列的。如果智能体试图从这些连续的经验中学习，就可能因为它们之间的强相关性而出现问题。为避免这一点，我们采用一种称为**经验回放**的技术，为训练智能体生成不相关的经验。经验回放的做法是：把智能体的经验（即智能体接收到的状态、动作和奖励）存入记忆缓冲区，然后从缓冲区中随机抽取一个小批量的经验用于学习。随着智能体与环境交互，经验元组 $(S_t, A_t, R_t, S_{t+1})$ 会在每个时间步被加入记忆缓冲区。

为方便起见，我们将把经验存储为具名元组（named tuple）。

In [ ]:
# 把经验存为具名元组
experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

通过使用经验回放，我们避免了有害的相关性、振荡和不稳定。此外，经验回放还让智能体有可能在多次权重更新中复用同一份经验，从而提高了数据效率。

<a name="7"></a>
## 7 - 带经验回放的深度 Q 学习算法

既然我们已经了解了将要使用的全部技术，就可以把它们组合起来，得到带经验回放的深度 Q 学习算法。
<br>
<br>
<figure>
  <img src = "图片/deep_q_algorithm.png" width = 90% style = "border: thin silver solid; padding: 0px">

> **图中文字（中文）**：算法 1：带经验回放的深度 Q 学习（Deep Q-Learning with Experience Replay） 1 初始化容量为 N 的记忆缓冲区 D 2 用随机权重初始化 Q 网络 3 用权重……初始化目标 Q 网络 4 for episode = 1 to … do     接收初始观测状态 S1 5   for t = 1 to … do 6     观测状态 St，并用 ε-贪心策略选择动作 At 7     在环境中执行动作 At，获得奖励 Rt 和下一状态 St+1 8     把经验元组 (St, At, Rt, St+1) 存入记忆缓冲区 D 9     每隔 C 步执行一次学习更新： 10        从 D 中随机抽取一个小批量的经验元组 (Sj, Aj, Rj, Sj+1) 11        若该回合在第 j+1 步终止，则令 yj = Rj；否则令 yj = Rj + γ·max_a' Q(sj+1, a') 12        针对 Q 网络权重，对 (yj − Q(sj, aj))² 执行一步梯度下降 13        用软更新（soft update）更新 Q 网络的权重     end end
      <figcaption style = "text-align: center; font-style: italic">图 3. 带经验回放的深度 Q 学习。</figcaption>
</figure>

<a name="ex02"></a>
### 练习 2

在本练习中，你将实现上面*图 3*所示算法的第 ***12*** 行，并计算 $y$ 目标与 $Q(s,a)$ 值之间的损失。在下面的单元格中，补全 `compute_loss` 函数，把 $y$ 目标设为：

$$
\begin{equation}
    y_j =
    \begin{cases}
      R_j & \text{if episode terminates at step  } j+1\\
      R_j + \gamma \max_{a'}\hat{Q}(s_{j+1},a') & \text{otherwise}\\
    \end{cases}       
\end{equation}
$$

这里有几处需要注意：

* `compute_loss` 函数接收一个小批量的经验元组。这个小批量经验元组会被解包，以取出 `states`、`actions`、`rewards`、`next_states` 和 `done_vals`。你应记住，这些变量是*TensorFlow 张量*，其大小取决于小批量的大小。例如，若小批量大小为 `64`，则 `rewards` 和 `done_vals` 都是含 `64` 个元素的 TensorFlow 张量。


* 当变量是含许多元素的张量时，用 `if/else` 语句设置 $y$ 目标是行不通的。不过请注意，你可以用 `done_vals` 把上述逻辑写成一行代码。为此，回忆一下 `done` 是一个布尔变量：当回合在第 $j+1$ 步终止时取值为 `True`，否则为 `False`。考虑到布尔值 `True` 的数值为 `1`、布尔值 `False` 的数值为 `0`，你可以用因子 `(1 - done_vals)` 把上述逻辑写成一行代码。这里有个提示：注意当 `done_vals` 为 `True` 时 `(1 - done_vals)` 的值为 `0`，当 `done_vals` 为 `False` 时其值为 `1`。

最后，通过计算 `y_targets` 与 `q_values` 之间的均方误差（`MSE`）来求损失。计算均方误差时，你应使用已经导入的程序包 `MSE`：

```python
from tensorflow.keras.losses import MSE
```


In [ ]:
# UNQ_C2
# 计分函数：calculate_loss

def compute_loss(experiences, gamma, q_network, target_q_network):
    """计算损失。
   
    参数：
      experiences: (tuple) 由 ["state", "action", "reward", "next_state", "done"] 具名元组组成的元组
      gamma: (float) 折扣因子。
      q_network: (tf.keras.Sequential) 用于预测 q 值的 Keras 模型
      target_q_network: (tf.keras.Sequential) 用于预测目标的 Keras 模型
         
    返回：
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) y 目标值与 Q(s,a) 值之间的均方误差。"""
    
    # 解包经验元组的小批量
    states, actions, rewards, next_states, done_vals = experiences
    
    # 计算 max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # 若回合终止则设 y = R，否则设 y = R + γ max Q^(s,a)。
    ### 在此处编写代码 ###
    y_targets = None
    ### 代码编写结束 ###
    
    # 获取 q 值
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
        
    # 计算损失
    ### 在此处编写代码 ###
    loss = None 
    ### 代码编写结束 ###
    
    return loss

In [ ]:
# 单元测试
test_compute_loss(compute_loss)

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    

```python
def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Karas model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """

    
    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))
    
    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
    
    # Calculate the loss
    loss = MSE(y_targets, q_values)
    
    return loss

```
 
    

<a name="8"></a>
## 8 - 更新网络权重

我们将用下面的 `agent_learn` 函数实现[图 3](#7)所示算法的第 ***12 -14*** 行。`agent_learn` 函数将用自定义训练循环更新 $Q$ 网络和目标 $\hat Q$ 网络的权重。由于我们使用自定义训练循环，需要通过 `tf.GradientTape` 实例取得梯度，然后调用 `optimizer.apply_gradients()` 更新 $Q$ 网络的权重。注意我们还使用了 `@tf.function` 装饰器来提升性能。没有这个装饰器，训练时间会翻倍。若想进一步了解如何用 `@tf.function` 提升性能，请参阅 [TensorFlow 文档](https://www.tensorflow.org/guide/function)。

该函数的最后一行用[软更新](#6.1)更新目标 $\hat Q$ 网络的权重。若想知道它在代码中是如何实现的，我们建议你看一看 `utils` 模块中的 `utils.update_target_network` 函数。

In [ ]:
@tf.function
def agent_learn(experiences, gamma):
    """更新 Q 网络的权重。
   
    参数：
      experiences: (tuple) 由 ["state", "action", "reward", "next_state", "done"] 具名元组组成的元组
      gamma: (float) 折扣因子。"""
    
    # 计算损失
    with tf.GradientTape() as tape:
        loss = compute_loss(experiences, gamma, q_network, target_q_network)

    # 获取损失对权重的梯度。
    gradients = tape.gradient(loss, q_network.trainable_variables)
    
    # 更新 q_network 的权重。
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

    # 更新目标 q_network 的权重
    utils.update_target_network(q_network, target_q_network)

<a name="9"></a>
## 9 - 训练智能体

现在我们准备好训练智能体来解决月球着陆器环境了。在下面的单元格中，我们将逐行实现[图 3](#7)中的算法（请注意，我们在下面附上了同一算法以便查阅，免得你在 notebook 中来回滚动）：

* **第 1 行**：我们用容量 $N =$ `MEMORY_SIZE` 初始化 `memory_buffer`。注意我们用 `deque` 作为 `memory_buffer` 的数据结构。


* **第 2 行**：这一行跳过，因为我们已在[练习 1](#ex01)中初始化了 `q_network`。


* **第 3 行**：我们通过把 `target_q_network` 的权重设为与 `q_network` 相同来初始化它。


* **第 4 行**：我们开始外层循环。注意我们设 $M =$ `num_episodes = 2000`。这个数值是合理的，因为使用本 notebook 的默认参数，智能体应能在不到 `2000` 个回合内解决月球着陆器环境。


* **第 5 行**：我们用 `.reset()` 方法把环境重置到初始状态并取得初始状态。


* **第 6 行**：我们开始内层循环。注意我们设 $T =$ `max_num_timesteps = 1000`。这意味着若回合在 `1000` 个时间步后仍未终止，就会自动终止。


* **第 7 行**：智能体观测当前的 `state`，并用 $\epsilon$-贪心策略选择一个 `action`。我们的智能体起初使用 $\epsilon =$ `epsilon = 1`，此时 $\epsilon$-贪心策略等价于等概率随机策略。这意味着训练初期，无论观测到什么 `state`，智能体都只是随机采取动作。随着训练推进，我们将用给定的 $\epsilon$ 衰减率把 $\epsilon$ 缓慢降到最小值。我们希望这个最小值接近零，因为 $\epsilon = 0$ 时 $\epsilon$-贪心策略等价于贪心策略。这意味着训练后期，智能体会倾向于选择它认为（基于过往经验）能使 $Q(s,a)$ 最大的 `action`。我们把 $\epsilon$ 的最小值设为 `0.01` 而非恰好 0，因为我们总希望在训练中保留一点探索。若想知道它在代码中是如何实现的，我们建议你看一看 `utils` 模块中的 `utils.get_action` 函数。


* **第 8 行**：我们用 `.step()` 方法在环境中采取给定的 `action`，并获得 `reward` 和 `next_state`。


* **第 9 行**：我们把 `experience(state, action, reward, next_state, done)` 元组存入 `memory_buffer`。注意我们还存储了 `done` 变量，以便跟踪回合何时终止。这让我们能在[练习 2](#ex02)中设置 $y$ 目标。


* **第 10 行**：我们检查是否满足执行学习更新的条件。我们用自定义的 `utils.check_update_conditions` 函数来完成。该函数检查是否已过去 $C =$ `NUM_STEPS_FOR_UPDATE = 4` 个时间步，以及 `memory_buffer` 中是否有足够的经验元组填满一个小批量。例如，若小批量大小为 `64`，则 `memory_buffer` 中至少要有 `64` 个经验元组才能满足后一个条件。若条件满足，`utils.check_update_conditions` 函数返回 `True`，否则返回 `False`。


* **第 11 - 14 行**：若 `update` 变量为 `True`，我们执行一次学习更新。学习更新包括：从 `memory_buffer` 中随机抽取一个小批量的经验元组，设置 $y$ 目标，执行梯度下降，并更新网络的权重。后三步我们将用[第 8 节](#8)中定义的 `agent_learn` 函数完成。


* **第 15 行**：在内层循环每次迭代结束时，我们把 `next_state` 设为新的 `state`，以便循环能从这个新状态重新开始。此外，我们检查回合是否到达终止状态（即检查 `done = True`）。若已到达终止状态，就跳出内层循环。


* **第 16 行**：在外层循环每次迭代结束时，我们更新 $\epsilon$ 的值，并检查环境是否已被解决。若智能体在最近 `100` 个回合中平均得到 `200` 分，我们就认为环境已被解决。若环境尚未解决，就继续外层循环并开始新回合。

最后，我们要说明，我们还加入了几个额外变量来跟踪智能体在每个回合中得到的累计分数。这有助于判断智能体是否已解决环境，也让我们能看到智能体在训练期间的表现。我们还用 `time` 模块测量训练耗时。

<br>
<br>
<figure>
  <img src = "图片/deep_q_algorithm.png" width = 90% style = "border: thin silver solid; padding: 0px">

> **图中文字（中文）**：算法 1：带经验回放的深度 Q 学习（Deep Q-Learning with Experience Replay） 1 初始化容量为 N 的记忆缓冲区 D 2 用随机权重初始化 Q 网络 3 用权重……初始化目标 Q 网络 4 for episode = 1 to … do     接收初始观测状态 S1 5   for t = 1 to … do 6     观测状态 St，并用 ε-贪心策略选择动作 At 7     在环境中执行动作 At，获得奖励 Rt 和下一状态 St+1 8     把经验元组 (St, At, Rt, St+1) 存入记忆缓冲区 D 9     每隔 C 步执行一次学习更新： 10        从 D 中随机抽取一个小批量的经验元组 (Sj, Aj, Rj, Sj+1) 11        若该回合在第 j+1 步终止，则令 yj = Rj；否则令 yj = Rj + γ·max_a' Q(sj+1, a') 12        针对 Q 网络权重，对 (yj − Q(sj, aj))² 执行一步梯度下降 13        用软更新（soft update）更新 Q 网络的权重     end end
      <figcaption style = "text-align: center; font-style: italic">图 4. 带经验回放的深度 Q 学习。</figcaption>
</figure>
<br>

**注意：** 使用本 notebook 的默认参数，下面这个单元格运行需要 10 到 15 分钟。

In [ ]:
start = time.time()

num_episodes = 2000
max_num_timesteps = 1000

total_point_history = []

num_p_av = 100    # 用于求平均的总点数
epsilon = 1.0     # ε-贪心策略的初始 ε 值

# 创建容量为 N 的记忆缓冲区 D
memory_buffer = deque(maxlen=MEMORY_SIZE)

# 把目标网络的权重设为与 Q 网络相同
target_q_network.set_weights(q_network.get_weights())

for i in range(num_episodes):
    
    # 把环境重置到初始状态并获取初始状态
    state = env.reset()
    total_points = 0
    
    for t in range(max_num_timesteps):
        
        # 从当前状态 S 出发，用 ε-贪心策略选择一个动作 A
        state_qn = np.expand_dims(state, axis=0)  # 状态的形状必须符合 q_network 的要求
        q_values = q_network(state_qn)
        action = utils.get_action(q_values, epsilon)
        
        # 执行动作 A，获得奖励 R 和下一状态 S'
        next_state, reward, done, _ = env.step(action)
        
        # 把经验元组 (S,A,R,S') 存入记忆缓冲区。
        # 为方便起见，我们也保存 done 变量。
        memory_buffer.append(experience(state, action, reward, next_state, done))
        
        # 每经过 NUM_STEPS_FOR_UPDATE 个时间步才更新一次网络。
        update = utils.check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)
        
        if update:
            # 从 D 中随机采样一个经验元组小批量 (S,A,R,S')
            experiences = utils.get_experiences(memory_buffer)
            
            # 设置 y 目标，执行一步梯度下降，
            # 并更新网络权重。
            agent_learn(experiences, GAMMA)
        
        state = next_state.copy()
        total_points += reward
        
        if done:
            break
            
    total_point_history.append(total_points)
    av_latest_points = np.mean(total_point_history[-num_p_av:])
    
    # 更新 ε 值
    epsilon = utils.get_new_eps(epsilon)

    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}", end="")

    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}")

    # 我们认为，如果得到
    # 最后 100 个回合中 200 个点的平均值。
    if av_latest_points >= 200.0:
        print(f"\n\nEnvironment solved in {i+1} episodes!")
        q_network.save('lunar_lander_model.h5')
        break
        
tot_time = time.time() - start

print(f"\nTotal Runtime: {tot_time:.2f} s ({(tot_time/60):.2f} min)")

我们可以把分数历史画出来，看看智能体在训练期间如何进步。

In [ ]:
# 绘制点历史
utils.plot_history(total_point_history)

<a name="10"></a>
## 10 - 观察训练后的智能体

现在我们已经训练好智能体，可以看看它的表现了。我们将用 `utils.create_video` 函数，生成一段智能体使用训练好的 $Q$ 网络与环境交互的视频。`utils.create_video` 函数使用 `imageio` 库生成视频。该库会产生一些干扰性的警告，为抑制这些警告，我们运行下面的代码。

In [ ]:
# 屏蔽来自 imageio 的警告
import logging
logging.getLogger().setLevel(logging.ERROR)

在下面的单元格中，我们生成一段智能体使用训练好的 `q_network` 与月球着陆器环境交互的视频。视频以给定的 `filename` 保存到 `videos` 文件夹。我们用 `utils.embed_mp4` 函数把视频嵌入 Jupyter Notebook，这样就能在这里直接观看，无需下载。

需要说明的是，由于月球着陆器开始时其质心会被施加一个随机初始力，每次运行下面的单元格你都会看到不同的视频。如果智能体训练得当，无论初始力如何，它都应能把月球着陆器降落在着陆平台上。

In [ ]:
filename = "./视频/lunar_lander.mp4"

utils.create_video(filename, env, q_network)
utils.embed_mp4(filename)

<a name="11"></a>
## 11 - 恭喜！

你已成功使用带经验回放的深度 Q 学习，训练出一个能把月球着陆器安全降落在月球表面着陆平台上的智能体。恭喜！

<a name="12"></a>
## 12 - 参考文献

若想进一步了解深度 Q 学习，我们推荐你阅读以下论文。


* [Human-level Control Through Deep Reinforcement Learning](https://storage.googleapis.com/deepmind-media/dqn/DQNNaturePaper.pdf)


* [Continuous Control with Deep Reinforcement Learning](https://arxiv.org/pdf/1509.02971.pdf)


* [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)